<a href="https://colab.research.google.com/github/ZahraRasooli-200/MATLAB-practice/blob/main/cat_dog_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import os
import tarfile
import urllib.request
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import matplotlib.pyplot as plt

In [28]:
#function to download and extract dataset
def download_and_extract(url, dest):
    if not os.path.exists(dest):
        os.makedirs(dest)
    file_name = url.split('/')[-1]
    file_path = os.path.join(dest, file_name)
    if not os.path.exists(file_path):
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, file_path)
    print(f"Extracting {file_path}...")
    with tarfile.open(file_path) as tar:
        tar.extractall(dest)

In [29]:
root = '/data'
images_url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz'
annotations_url = 'https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz'
download_and_extract(images_url, root)
download_and_extract(annotations_url, root)

Extracting /data/images.tar.gz...


/tmp/ipykernel_500/1755157291.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest)


Extracting /data/annotations.tar.gz...


In [30]:
def get_image_paths_labels(images_path, annotations_path):
    image_paths = []
    labels = []
    class_map = {
        'Persian': 0,
        'shiba_inu': 1,
        'saint_bernard': 2,
    }
    with open(annotations_path) as f:
        lines = f.readlines()
    for line in lines:
        img_name = line.strip().split()[0]
        breed_name = img_name.split('_',1)[0]
        if breed_name in class_map:
            img_path = os.path.join(images_path, f'{img_name}.jpg')
            if os.path.exists(img_path):
                image_paths.append(img_path)
                labels.append(class_map[breed_name])
    return image_paths, labels

In [31]:
#split using Pytorch random_split on indices
image_paths, labels = get_image_paths_labels(images_path='./data/images', annotations_path='/data/annotations/trainval.txt')
indices = list(range(len(image_paths)))
train_indices, val_indices = random_split(indices,[0.8,0.2],generator=torch.Generator().manual_seed(42))

train_paths = [image_paths[i] for i in train_indices]
train_labels = [labels[i] for i in train_indices]
val_path = [image_paths[i] for i in val_indices]
val_labels = [labels[i] for i in val_indices]

/tmp/ipykernel_500/1040958343.py:4: UserWarning: Length of split at index 0 is 0. This might result in an empty dataset.
  train_indices, val_indices = random_split(indices,[0.8,0.2],generator=torch.Generator().manual_seed(42))
/tmp/ipykernel_500/1040958343.py:4: UserWarning: Length of split at index 1 is 0. This might result in an empty dataset.
  train_indices, val_indices = random_split(indices,[0.8,0.2],generator=torch.Generator().manual_seed(42))


In [32]:
class CustomPetDataset(Dataset):
  def __init__(self, image_paths, labels, transform):
    self.image_paths = image_paths
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    img_path = self.image_paths[idx]
    image = Image.open(img_path).convert('RGB')
    label = self.labels[idx]
    image = self.transform(image)
    return image, label

In [33]:
#Data transformation (using RGB,larger size,and standard augmentations)
train_transform = transforms.Compose([
    transforms.Resize((256)), #128x256x3 --> 256x512x3
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), #3x224x224\
])

val_transform = transforms.Compose([
    transforms.Resize((256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

In [35]:
#Create separate datasets with split lists and transforms
train_dataset = CustomPetDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CustomPetDataset(val_path, val_labels, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [40]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Sequential( #input Nx3x224x224
    nn.Conv2d(3,16,Kernel_size=3, padding=1),
    nn.ReLU(),#Nx16x224x224
    nn.MaxPool2d(2),#Nx16x112x112
    nn.Conv2d(16,32,Kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),#Nx32x56x56
    nn.Conv2d(32,64,Kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),#nx64x28x28
    nn.Flatten()
)
    self.fc = nn.Sequential(
      nn.Linear(64*28*28,128),
      nn.ReLU(),
      nn.Linear(128,3)
)
  def forward(self, x):
    x = self.conv(x)
    x = self.fc(x)
    return x


In [41]:
#Device configuration
Device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [42]:
def build_model():
  model = SimpleCNN().to(Device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
  return model, criterion, optimizer

In [43]:
def train_epoch(model, loader, criterion, optimizer, device):
  model.train()
  running_loss = 0.0
  corrects = 0
  total = 0
  for images, targets in loader:
    images = images.to(device)
    targets = targets.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
    predictions = outputs.argmax(dim=1)
    corrects += (predictions == targets).sum().item()
    total += targets.size(0)
  train_loss /= len(loader)
  train_acc = corrects / total

  return train_loss, train_acc


In [44]:
def val_epoch(model, loader, criterion, device):
  model.eval()
  running_loss = 0.0
  corrects = 0
  total = 0
  with torch.no_grad():
    for images, targets in loader:
      images = images.to(device)
      targets = targets.to(device)
      outputs = model(images)
      loss = criterion(outputs, targets)
      val_loss += loss.item()
      predictions = outputs.argmax(dim=1)
      corrects += (predictions == targets).sum().item()
      total += targets.size(0)
    val_loss /= len(loader)
    train_acc = corrects / total

    return train_loss, train_acc


In [ ]:
#BUILD MODEL
model, criterion, optimizer = build_model()
epochs = 100
best_acc = 0.0
for epoch in range(epochs):
  train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, Device)
  val_loss, val_acc = val_epoch(model, val_loader, criterion, Device)
  if val_acc > best_acc:
    best_acc = val_acc
    torch.save(model.state_dict(), 'best_model.pth')
  print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

In [ ]:
#LOAD BEST MODEL
model.load_state_dict(torch.load('best_model.pth'))
model.eval()